In [1]:
import sys
sys.path.append('..')

import os
os.environ["KERAS_BACKEND"] = "torch"

import time
import keras
import numpy as np
# Necesario para TextVectorization y tf.data.
import tensorflow as tf
from models.training import compile_model, get_callbacks
from config.settings import Settings
from features.embeddings import load_gensim_embeddings
from datasets.dataset import create_dataset
from features.vectorizer import TextVectorizerModel
from datasets.loader import load_splits, save_json
from models.siamese_lstm import SiameseLSTM
from datasets.paths import ProjectPaths
from gensim.models import KeyedVectors


c:\Users\malos\Documents\GitHub\JustShare\server\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
print(keras.config.backend())


torch


In [3]:
settings = Settings()
print(settings)


batch_size=64 mlp_dropout=0.4 lstm_dropout=0.3 pooling='mean' similarity='cosine' hidden_dim=64 bidirectional=False mlp_layers=[32] concat_features=['diff'] epochs=20 augmented_data=True siamese_name='lstm_mean_cosine' max_len=26 augmented_max_len=26 case=False strip_punctuation=True


In [4]:
paths = ProjectPaths(siamese_name=settings.siamese_name)


In [5]:
if settings.augmented_data:
	max_len = settings.augmented_max_len
	train_dir = paths.augmented_dir
else:
	max_len = settings.max_len
	train_dir = paths.processed_dir

print(max_len)

26


In [6]:
splits = {
	"train": train_dir,
	"dev": paths.processed_dir,
    "test": paths.processed_dir
}

datasets = load_splits(splits)

train_df = datasets["train"]
dev_df = datasets["dev"]
test_df = datasets["test"]


In [7]:
display(train_df)


,sentence1,sentence2,score,split,score_norm
0,Un avión está despegando.,Un avión está despegando.,5.0,train,1.00
1,Un aeroplano está despegando.,Un aeroplano está despegando.,5.0,train,1.00
2,Un hombre está tocando una gran flauta.,Un hombre está tocando una flauta.,3.8,train,0.76
3,Un marido está tocando una gran flauta.,Un hombre está tocando una flauta.,3.8,train,0.76
4,Un hombre está untando queso rallado en una pi...,Un hombre está untando queso rallado en una pi...,3.8,train,0.76
...,...,...,...,...,...
10534,El Presidente se dirige a Bahrein,Presidente Xi: China seguirá ayudando a combat...,0.0,train,0.00
10535,El Director se dirige a Bahrein,Presidir Xi: China seguirá ayudando a luchar e...,0.0,train,0.00
10536,China y la India se comprometen a fomentar los...,China lucha por tranquilizar a los nerviosos c...,0.0,train,0.00
10537,El portavoz de Putin: Los cargos por dopaje pa...,Lo último en clima severo: 1 muerto en Texas d...,0.0,train,0.00


In [8]:
print("Train length:", len(train_df))
print("Dev length:", len(dev_df))


Train length: 10539
Dev length: 1497


In [9]:
all_sentences = list(train_df["sentence1"]) + list(train_df["sentence2"])

vectorizer = TextVectorizerModel(
    max_len=max_len,
    case=settings.case,
    strip_punctuation=settings.strip_punctuation
)

vectorizer.adapt(all_sentences)

vocab = vectorizer.get_vocabulary()
word2idx = {word: idx for idx, word in enumerate(vocab)}
print(f"Vocabulary size: {len(vocab)}")

vectorizer.save(paths.vectorizer_dir)


Vocabulary size: 15276


c:\Users\malos\Documents\GitHub\JustShare\server\.venv\Lib\site-packages\keras\src\saving\saving_api.py:107: UserWarning: You are saving a model that has not yet been built. It might not contain any weights yet. Consider building the model first by calling it on some data.
  return saving_lib.save_model(model, filepath)


In [10]:
# https://github.com/aitoralmeida/spanish_word2vec
wv = KeyedVectors.load_word2vec_format(paths.word2vec_path, binary=True)


In [11]:
dim = wv.vector_size
print(dim)
print(len(wv))


400
1943871


In [12]:
vocab = vectorizer.vectorizer.get_vocabulary()
print(vocab[:10])


['', '[UNK]', np.str_('de'), np.str_('la'), np.str_('el'), np.str_('en'), np.str_('un'), np.str_('una'), np.str_('a'), np.str_('los')]


In [13]:
embedding_matrix, missing_words = load_gensim_embeddings(wv, word2idx, dim)

print(embedding_matrix.shape)

print(missing_words[:50])


Encontradas: 14863/15274 (97.31%)
No encontradas: 411/15274 (2.69%)
(15276, 400)
[np.str_('ixic'), np.str_('neener'), np.str_('us30ytrr'), np.str_('us10ytrr'), np.str_('promorsi'), np.str_('lendingtree'), np.str_('chiqelo'), np.str_('sorenstam'), np.str_('sistánbaluchistán'), np.str_('inglésturco'), np.str_('hakimullah'), np.str_('152015'), np.str_('mh17'), np.str_('Äôs'), np.str_('strier'), np.str_('saferworld'), np.str_('prorusia'), np.str_('someoen'), np.str_('shalgam'), np.str_('x86'), np.str_('wolfcale'), np.str_('weisselberg'), np.str_('waksal'), np.str_('w32sobigcmm'), np.str_('usvisit'), np.str_('torsella'), np.str_('tafb'), np.str_('syndia'), np.str_('studabaker'), np.str_('stonesoft'), np.str_('staffenberg'), np.str_('sriyanto'), np.str_('solinvictus'), np.str_('smeone'), np.str_('shereka'), np.str_('rs24'), np.str_('ropeik'), np.str_('romeril'), np.str_('reutersipsos'), np.str_('promursi'), np.str_('pribbenow'), np.str_('polipíldora'), np.str_('plofsky'), np.str_('pashtoon')

In [14]:
np.save(paths.embedding_path, embedding_matrix)


In [15]:
train_dataset = create_dataset(train_df, vectorizer, settings.batch_size, shuffle=True)
dev_dataset = create_dataset(dev_df, vectorizer, settings.batch_size)


In [16]:
for (sent1, sent2), y in train_dataset.take(1):
	print("sent1:", sent1.shape)
	print("sent2:", sent2.shape)
	print("y:", y.shape)


sent1: (64, 26)
sent2: (64, 26)
y: (64,)


In [17]:
model = SiameseLSTM(
	vocab_size=len(vocab),
	embedding_dim=dim,
	hidden_dim=settings.hidden_dim,
	mlp_dropout=settings.mlp_dropout,
	lstm_dropout=settings.lstm_dropout,
	embedding_matrix=embedding_matrix,
	pooling=settings.pooling,
	similarity=settings.similarity,
	mlp_layers=settings.mlp_layers,
	bidirectional=settings.bidirectional,
	concat_features=settings.concat_features,
    name=settings.siamese_name
)


In [18]:
if model.mlp:
	model.mlp.summary()


In [19]:
head_model = model.get_head_model()
head_model.summary()


Model: "siamese_head"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 400) │  6,110,400 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cast (Cast)         │ (None, None)      │          0 │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, None, 64)  │    119,040 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (None, None, 1)   │          0 │ cast[0][0]        │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, None, 64)  │          0 │ lstm[0][0],       │
│                     │                   │            │ expand_dims[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sum_1 (Sum)         │ (None, 1)         │          0 │ expand_dims[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sum (Sum)           │ (None, 64)        │          0 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 1)         │          0 │ sum_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ true_divide         │ (None, 64)        │          0 │ sum[0][0],        │
│ (TrueDivide)        │                   │            │ add[0][0]         │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,229,440 (23.76 MB)

 Trainable params: 119,040 (465.00 KB)

 Non-trainable params: 6,110,400 (23.31 MB)

In [20]:
dummy_sent1 = tf.zeros((1, max_len), dtype=tf.int32)
dummy_sent2 = tf.zeros((1, max_len), dtype=tf.int32)

model((dummy_sent1, dummy_sent2))

model.summary()


Model: "lstm_mean_cosine"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 400)      │     6,110,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 64)       │       119,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,229,440 (23.76 MB)

 Trainable params: 119,040 (465.00 KB)

 Non-trainable params: 6,110,400 (23.31 MB)

In [21]:
model = compile_model(model)

callbacks = get_callbacks(paths.siamese_path)


In [22]:
weights = model.embedding.get_weights()[0]

for i in range(10):
    print(i, vocab[i], weights[i][:5])
    

0  [0. 0. 0. 0. 0.]
1 [UNK] [-0.29798955  0.08085851  0.0722797   0.20451036  0.16621761]
2 de [ 0.21769126 -2.2936897  -1.3649052  -2.588661    2.8798313 ]
3 la [-1.8710854  -0.46249285  0.8997163  -0.17077468  2.4137783 ]
4 el [ 0.40440384 -2.3295271  -5.4817753   0.2512127   0.8362291 ]
5 en [ 1.6279118e+00  1.9744599e-04 -4.8397598e+00 -1.7532663e-01
  3.0406034e+00]
6 un [ 1.8357593   0.28429332 -3.9700549   0.31613564  1.1295024 ]
7 una [-0.02631674  1.976948    0.06162561 -1.074304    1.8378303 ]
8 a [-1.2744071   0.12961593 -1.9868547   0.04239245  3.7093022 ]
9 los [-0.8855123 -3.2326047 -0.5929021 -3.0912414  1.3124015]


In [23]:
start_time = time.perf_counter()

history = model.fit(
	train_dataset,
	validation_data=dev_dataset,
	epochs=settings.epochs,
	callbacks=callbacks
)

train_time = time.perf_counter() - start_time 

np.save(paths.history_path, history.history)


Epoch 1/20
165/165 ━━━━━━━━━━━━━━━━━━━━ 43s 257ms/step - loss: 0.0687 - mae: 0.2150 - rmse: 0.2621 - val_loss: 0.1107 - val_mae: 0.2705 - val_rmse: 0.3327 - learning_rate: 0.0010
Epoch 2/20
165/165 ━━━━━━━━━━━━━━━━━━━━ 41s 246ms/step - loss: 0.0550 - mae: 0.1909 - rmse: 0.2346 - val_loss: 0.0878 - val_mae: 0.2396 - val_rmse: 0.2963 - learning_rate: 0.0010
Epoch 3/20
165/165 ━━━━━━━━━━━━━━━━━━━━ 40s 241ms/step - loss: 0.0479 - mae: 0.1767 - rmse: 0.2189 - val_loss: 0.0934 - val_mae: 0.2464 - val_rmse: 0.3057 - learning_rate: 0.0010
Epoch 4/20
165/165 ━━━━━━━━━━━━━━━━━━━━ 40s 239ms/step - loss: 0.0426 - mae: 0.1665 - rmse: 0.2065 - val_loss: 0.0879 - val_mae: 0.2382 - val_rmse: 0.2964 - learning_rate: 0.0010
Epoch 5/20
165/165 ━━━━━━━━━━━━━━━━━━━━ 41s 247ms/step - loss: 0.0383 - mae: 0.1573 - rmse: 0.1957 - val_loss: 0.0861 - val_mae: 0.2349 - val_rmse: 0.2935 - learning_rate: 0.0010
Epoch 6/20
165/165 ━━━━━━━━━━━━━━━━━━━━ 40s 242ms/step - loss: 0.0353 - mae: 0.1512 - rmse: 0.1878 - val_

In [24]:
run_config = {
    "sequence_length": max_len,
    "data_augmentation": settings.augmented_data,
    "train_time_s": train_time
}

save_json(run_config, paths.config_path)
